# Harmonise (step 8a): cohort, clock, units and cleaning rules

Turns the concept map into long tables, one parquet per site and concept group, in `derived/harmonised/` (row-level: stays in the SDE, git-ignored). Columns: site, stay_key, concept, hour (hours since the stay's t0, true elapsed time), value, value_text (text concepts), flag, source.

Rules (see `DATA_ISSUES.md` and the `harmonise.py` docstring): stays over 6 hours only; all chart times to UTC before subtracting t0 (SMH chart times are UTC, the rest London local); haemoglobin to g/L; haematocrit to percent; placeholder zeros to missing; GOSH diastolic equal to systolic to missing; GOSH base excess derived from pH and bicarbonate and flagged `derived`; tidal volume unit stripped; I:E ratio as the x of 1:x.

Fluids and drugs are not in this notebook (see `harmonise_fluids`, next step). Only counts and hours leave the SDE.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path
import pandas as pd
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import concepts as C
import harmonise as H

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
assembled_dir = open_picu_dir / "derived" / "assembled"
reports_dir = open_picu_dir / "reports"
harmonised_dir = open_picu_dir / "derived" / "harmonised"
static = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
today = date.today().isoformat()
co = H.cohort(static)
print(H.__file__)
print(co.groupby("site").size().to_dict(), "stays over 6 hours of", static.groupby("site").size().to_dict())

## 1. Is the clock right?
Median hours of the first heart rate after t0, by site and season. If the time zones are handled correctly, summer and winter should agree (an error would show as a one-hour step). The share before t0 or more than 24 h after shows how far chart times sit from the recorded admission.

In [ ]:
clock = H.time_alignment_check(assembled_dir, static)
clock

## 2. Harmonise every non-fluid, non-drug concept
Reads each source with its row times, applies the rules and writes one parquet per site and group. The report has counts only: rows read, outside the cohort, with a bad time, non-numeric, set missing, converted, derived, and the 5th, 50th and 95th percentile of `hour`. This is slow (millions of rows); run it once.

In [ ]:
# full run
report = H.harmonise_all(assembled_dir, static, harmonised_dir)
report.to_csv(Path(reports_dir) / "harmonise_report.csv", index=False)
report.drop(columns="cohort_stays")

In [ ]:
# update
report = H.harmonise_all(assembled_dir, static, harmonised_dir,
                         concepts=["ett_size", "ett_depth", "ett_route", "ett_cuff"])

## 2b. Fluids
Fluids have their own definitions (see the `harmonise.py` section header and `DATA_ISSUES.md` G8, S3, R6): hourly ml, with `intake_total` at all sites and `intake_iv` (infusion, colloids and drug volume; no blood, enteral or TPN) at RBH and SMH only. GOSH hourly amounts come from differencing the 24 h running totals before the Epic switch and from `caboodle_fluid_balance` after it. Nothing is removed: hours above the 99.9th percentile of ml per kg per hour (per site and concept) are flagged `outlier`, and the cap values are reported so the choice of percentile can be reviewed.

In [ ]:
fl_report, fl_caps = H.harmonise_fluids(assembled_dir, static, harmonised_dir)
fl_report.to_csv(Path(reports_dir) / "harmonise_fluids_report.csv", index=False)
fl_caps.to_csv(Path(reports_dir) / "harmonise_fluids_caps.csv", index=False)
display(fl_report[["site", "concept", "source", "stays", "rows", "rows_gap", "rows_outlier", "hour_median"]])
display(fl_caps)

## 2c. Do the harmonised rows lie inside the stay?
For every harmonised file, per site and concept: the share of rows more than 24 h before t0, from 24 h before up to t0, inside the stay (t0 to `los_hours`) and after the end of the stay. RBH blood gas and labs reach far before and after t0 (5th percentile about -1,600 h), which suggests samples from outside the stay. Nothing is removed here; this shows how much would be.

In [ ]:
window = H.window_check(harmonised_dir, static, assembled_dir=assembled_dir)
window.to_csv(Path(reports_dir) / "harmonise_window.csv", index=False)
window[(window["pct_before_24h"] > 1) | (window["pct_after_end"] > 1)].sort_values(["site", "pct_before_24h"], ascending=[True, False])

### 2d. RBH: why do the vitals run past the recorded end?
Uses the ward times in the RBH vitals table: ward in and out relative to t0, whether a stay key has several ward episodes or admission keys, and how the last heart rate compares with the static end and with the ward out. Also lists the clinical units that appear in the vitals, in case the rows after the end belong to another unit.

In [ ]:
rbh_end = H.rbh_end_check(assembled_dir, static)
print(rbh_end.to_string())
C.list_values(assembled_dir, "RBH", "ai_picu_vitals", "clinicalUnit")

**By clinical unit.** For each unit in the RBH vitals: rows, stays, and the percent of heart-rate rows more than 24 h before t0, from 24 h before to t0, inside the stay and after the static end. Shows which units the rows past the recorded end belong to.

In [ ]:
pd.set_option("display.width", 200)
H.rbh_unit_check(assembled_dir, static)

### 3. Vent mode values by site
Lists the distinct recorded ventilator mode strings per site (rows and stays, values seen in fewer than 10 stays hidden). This is the input for the vent mode name map. Counts only.

In [ ]:
for site in ["RBH", "SMH", "GOSH"]:
    C.concept_values(assembled_dir, "vent_mode", site, min_stays=10, top=40)

vent_values = {site: C.concept_values(assembled_dir, "vent_mode", site, min_stays=10, top=40)
               for site in ["RBH", "SMH", "GOSH"]}

### Distributions across sites
Per site and concept: quantiles of the harmonised values, share of zero, negative and flagged rows. Then a side-by-side comparison with flags where the sites' medians, spreads, tails or zero shares differ. Flags are places to look, not errors. Counts and ranges only; no minimum or maximum.

In [ ]:
dist = H.distribution_check(harmonised_dir)
cmp_ = H.compare_sites(dist)
print(len(cmp_), "concepts;", int((~cmp_["looks_ok"]).sum()), "flagged")
cmp_[~cmp_["looks_ok"]]

In [ ]:
cmp_[cmp_["looks_ok"]][["concept", "sites", "RBH", "SMH", "GOSH"]]
print(cmp_.to_string())

### Follow-up checks on flagged concepts
Looks inside the concepts flagged above: split by source, the most common high values (placeholders), the raw SMH I:E strings, and the GOSH potassium table columns (names only).

In [ ]:
print(H.source_check(harmonised_dir, "GOSH", "bg_venous_sat").to_string())
for site, concept, limit in [("GOSH", "insp_time", 5), ("GOSH", "albumin", 100), ("GOSH", "cvp", 40), ("GOSH", "potassium", 10)]:
    print(H.values_above(harmonised_dir, site, concept, limit).to_string(), "\n")
C.concept_values(assembled_dir, "ie_ratio", "SMH", min_stays=10, top=60)
print(pd.read_parquet(Path(assembled_dir)/"GOSH"/(C._L + "Potassium_merged.parquet")).columns.tolist())

In [ ]:
for col in ["unit_of_measure", "data_source"]:
    C.percentiles_by(assembled_dir, "GOSH", C._L + "Potassium_merged", "res_value", col)

   ### I:E ratio at SMH: what do the integers 11 to 19 mean?
   For each recorded value, the ratio implied by the set rate and inspiratory time at the same chart time. A recorded "14" with an implied median near 1.4 is 1.4; near 4 it is 1:4. Quantiles only.

In [ ]:
H.ie_check(harmonised_dir)

### 4. Drugs as recorded
Writes `{site}_drugs.parquet`: each drug record with its recorded value, unit, a `kind` read from the unit, route and the original text where a number was cut out of text. Nothing is converted or pooled. The same record in two tables is kept once. Counts only in the printed tables.

In [ ]:
drug_report, drug_units = H.harmonise_drugs(assembled_dir, static, harmonised_dir)
print(drug_report.groupby("site")[["rows", "rows_dup", "rows_outside_window", "rows_empty"]].sum())
drug_units.groupby(["site", "kind"])[["rows"]].sum()

Drug unit lookup. Lists every unit string recorded for drugs at each site (only those seen in at least 10 stays), with what kind of value it is and, where the string parses, a factor that converts the recorded value to mg (or units) per hour, or per kg per hour. Nothing in the harmonised data is converted. Strings that do not parse have an empty factor; check those first.

In [ ]:
lookup = H.drug_unit_lookup(harmonised_dir)
display(lookup[lookup["factor"].isna()][["site", "unit", "rows", "stays", "kind"]])
display(lookup[lookup["factor"].notna()])

## 5. Save the results for review

In [ ]:
flagged = window[(window["pct_before_24h"] > 1) | (window["pct_after_end"] > 1)]
cols = ["site", "concept", "stays", "rows", "rows_empty", "rows_non_numeric", "rows_set_missing", "rows_converted", "rows_derived", "rows_range_flag", "rows_outside_window", "rows_dup"]
rej = pd.concat([pd.read_parquet(Path(harmonised_dir) / f"{s}_rejected.parquet", columns=["site", "concept", "reason"]) for s in ["RBH", "SMH", "GOSH"]])
rej_counts = rej.groupby(["site", "concept", "reason"]).size().rename("rows").reset_index()
sizes = pd.Series({p.name: round(p.stat().st_size / 1e6, 1) for p in sorted(Path(harmonised_dir).glob("*.parquet"))}, name="MB").to_frame()
sections = {
    "Harmonise report (rows kept and rows changed, by site and concept)": report[[c for c in cols if c in report.columns]],
    "Rejected rows by reason": rej_counts,
    "Drug report (rows kept and dropped, by site and drug)": drug_report,
    "Drug units by site, drug, kind and unit": drug_units,
    "Output file sizes (MB)": sizes,
    "Fluids report": fl_report,
    "Fluids caps (ml per kg per hour)": fl_caps,
    "Window check (concepts with more than 1% of rows before -24 h or after the stay)": flagged,
    "Cross-site comparison": cmp_,
    "Distributions per site": dist.drop(columns=["flags"]).round(3),
}
out = Path(reports_dir) / f"harmonise_review_{today}.md"
out.write_text("\n\n".join(f"## {k}\n\n```\n{v.to_string()}\n```" for k, v in sections.items()) + "\n")
print("written:", out)

In [ ]:
r = report.assign(pct_removed=(100 * report["rows_outside_window"] / (report["rows"] + report["rows_outside_window"])).round(1))
print(r.groupby("site")[["rows", "rows_outside_window"]].sum())
r[r["pct_removed"] > 0.5][["site", "concept", "rows", "rows_outside_window", "pct_removed"]]

### Where do the rows of a stay lie compared with its recorded length?
Per stay, the last row against the recorded length of stay, and the first row against t0, for heart rate, SpO2, intake, pH and sodium. A few stays with a very long tail point to overlapping episodes under one stay key. A small excess in most stays points to a length of stay that is too short.

In [ ]:
span = H.stay_span_check(harmonised_dir, static)
span.to_csv(Path(reports_dir) / "harmonise_span.csv", index=False)
span

# Pool data

In [ ]:
pooled = H.load_pooled(harmonised_dir, groups=["vitals"], concepts=["hr", "spo2"])
drugs = H.load_pooled(harmonised_dir, drugs=True)